In [ ]:
!pip install -q numpy==1.26.4
!pip install -q monai==1.4.0

In [ ]:
!pip install -q --force-reinstall numpy==1.26.4

In [2]:
import numpy as np
import torch
import monai

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("monai:", monai.__version__)

numpy: 1.26.4
torch: 2.10.0+cu128
monai: 1.4.0


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import torch
import numpy as np
import urllib.request
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings("ignore")

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    Orientationd, NormalizeIntensityd,
    CropForegroundd, RandCropByPosNegLabeld,
    RandFlipd, RandShiftIntensityd, RandScaleIntensityd,
    RandRotate90d, ConcatItemsd, DeleteItemsd,
    Lambdad, ToTensord, AsDiscrete,
    RandGaussianNoised, RandAdjustContrastd,
    SpatialPadd
)
from monai.data import CacheDataset, DataLoader
from monai.networks.nets import SwinUNETR
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference

# ---- session timer ----
SESSION_START    = time.time()
TIME_BUDGET_SECS = 10 * 3600

def time_elapsed():   return time.time() - SESSION_START
def time_remaining(): return TIME_BUDGET_SECS - time_elapsed()
def budget_ok(buf=600): return time_remaining() > buf

# ---- device ----
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
scaler  = torch.cuda.amp.GradScaler(enabled=use_amp, init_scale=256.0)

torch.backends.cudnn.benchmark        = True
torch.backends.cudnn.deterministic    = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

print(f"Device: {device} | AMP: {use_amp}")

# ---- hyperparameters ----
FEATURE_SIZE  = 48
SPATIAL_SIZE  = (128, 128, 128)
NUM_SAMPLES   = 1
BATCH_SIZE    = 1
MAX_EPOCHS    = 70
WARMUP_EPOCHS = 5
VAL_INTERVAL  = 5
BASE_LR       = 1e-4
NUM_CLASSES   = 4
NUM_WORKERS   = 2
CASES_TO_USE  = 250
PATIENCE      = 15

CHECKPOINT_PATH    = "/kaggle/working/best_model.pth"
PRETRAINED_URL     = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/model_swinvit.pt"
PRETRAINED_PATH    = "/kaggle/working/model_swinvit.pt"

# ---- data ----
train_path = "/kaggle/input/datasets/dravyareddy/brats-2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

def get_file(case, keyword):
    for f in os.listdir(case):
        if keyword in f and (f.endswith(".nii") or f.endswith(".nii.gz")):
            path = os.path.join(case, f)
            if os.path.getsize(path) > 0:
                return path
    return None

cases = sorted(glob(os.path.join(train_path, "*")))
data  = []
for case in cases:
    flair = get_file(case, "t2f")
    t1    = get_file(case, "t1n")
    t1ce  = get_file(case, "t1c")
    t2    = get_file(case, "t2w")
    seg   = get_file(case, "seg")
    if None not in [flair, t1, t1ce, t2, seg]:
        data.append({"flair": flair, "t1": t1, "t1ce": t1ce, "t2": t2, "label": seg})

data = data[:CASES_TO_USE]
train_files, val_files = train_test_split(data, test_size=0.2, random_state=42)
print(f"Train: {len(train_files)} | Val: {len(val_files)}")

# ---- transforms ----
MODALITY_KEYS = ["flair", "t1", "t1ce", "t2"]

def brats_relabel(label):
    label[label == 4] = 3
    return label

train_transforms = Compose([
    LoadImaged(keys=MODALITY_KEYS + ["label"]),
    EnsureChannelFirstd(keys=MODALITY_KEYS + ["label"]),
    Lambdad(keys=["label"], func=brats_relabel),
    ConcatItemsd(keys=MODALITY_KEYS, name="image", dim=0),
    DeleteItemsd(keys=MODALITY_KEYS),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    SpatialPadd(keys=["image", "label"], spatial_size=SPATIAL_SIZE), 
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.3, max_k=3),
    RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
    RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
    RandGaussianNoised(keys="image", prob=0.15, mean=0.0, std=0.1),
    RandAdjustContrastd(keys="image", prob=0.15, gamma=(0.7, 1.5)),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=SPATIAL_SIZE,
        pos=3, neg=1,
        num_samples=NUM_SAMPLES,
    ),
    ToTensord(keys=["image", "label"])
])

val_transforms = Compose([
    LoadImaged(keys=MODALITY_KEYS + ["label"]),
    EnsureChannelFirstd(keys=MODALITY_KEYS + ["label"]),
    Lambdad(keys=["label"], func=brats_relabel),
    ConcatItemsd(keys=MODALITY_KEYS, name="image", dim=0),
    DeleteItemsd(keys=MODALITY_KEYS),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    SpatialPadd(keys=["image", "label"], spatial_size=SPATIAL_SIZE), 
    ToTensord(keys=["image", "label"])
])

# ---- datasets ----
print("Caching dataset...")
t0 = time.time()

train_ds = CacheDataset(train_files, train_transforms, cache_rate=0.6, num_workers=NUM_WORKERS)
val_ds   = CacheDataset(val_files,   val_transforms,   cache_rate=0.6,  num_workers=NUM_WORKERS)

print(f"Cache done in {(time.time()-t0)/60:.1f} min")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)
val_loader = DataLoader(
    val_ds, batch_size=1,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)

# ---- model ----
model = SwinUNETR(
    img_size=SPATIAL_SIZE,
    in_channels=4,
    out_channels=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    use_checkpoint=False,
    spatial_dims=3,
    dropout_path_rate=0.1,
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ---- pretrained weights ----
try:
    if not os.path.exists(PRETRAINED_PATH):
        print("Downloading pretrained weights...")
        urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
    model.load_from(weights=torch.load(PRETRAINED_PATH, map_location="cpu"))
    print("Pretrained encoder loaded")
except Exception as e:
    print(f"Pretrained load failed ({e}). Proceeding without.")

# ---- loss / optimizer / scheduler ----
loss_function = DiceFocalLoss(
    to_onehot_y=True,
    softmax=True,
    include_background=False,
    gamma=2.0,
    lambda_dice=1.0,
    lambda_focal=1.0,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-5, eps=1e-5)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr          = BASE_LR * 10,
    epochs          = MAX_EPOCHS,
    steps_per_epoch = len(train_loader),
    pct_start       = WARMUP_EPOCHS / MAX_EPOCHS,
    anneal_strategy = "cos",
    div_factor      = 10,
    final_div_factor = 1e4,
)

dice_metric = DiceMetric(include_background=False, reduction="mean")
post_pred   = AsDiscrete(argmax=True, to_onehot=NUM_CLASSES)
post_label  = AsDiscrete(to_onehot=NUM_CLASSES)

# ---- training loop ----
best_dice  = 0.0
no_improve = 0
nan_count  = 0

print(f"\nTraining: {MAX_EPOCHS} epochs | Budget: {TIME_BUDGET_SECS/3600:.1f} hrs")
print(f"SwinUNETR-{FEATURE_SIZE} | Crop: {SPATIAL_SIZE} | Cases: {CASES_TO_USE}\n")

for epoch in range(MAX_EPOCHS):

    if not budget_ok():
        print(f"Budget exhausted at epoch {epoch+1}.")
        break

    t_epoch   = time.time()
    epoch_loss    = 0.0
    valid_batches = 0

    model.train()
    for batch in train_loader:
        inputs = batch["image"].to(device, non_blocking=True).float()
        labels = batch["label"].to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(inputs)
            loss    = loss_function(outputs, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss    += loss.item()
        valid_batches += 1

    avg_loss = epoch_loss / max(valid_batches, 1)
    print(
        f"Epoch {epoch+1:03d}/{MAX_EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
        f"Epoch: {(time.time()-t_epoch)/60:.1f} min | "
        f"Elapsed: {time_elapsed()/60:.1f} min | "
        f"Left: {time_remaining()/60:.1f} min"
    )

    if (epoch + 1) % VAL_INTERVAL != 0:
        continue

    model.eval()
    dice_metric.reset()
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for val in val_loader:
            val_inputs = val["image"].to(device, non_blocking=True).float()
            val_labels = val["label"].to(device, non_blocking=True).long()

            with torch.cuda.amp.autocast(enabled=use_amp):
                val_outputs = sliding_window_inference(
                    val_inputs,
                    roi_size      = SPATIAL_SIZE,
                    sw_batch_size = 2,
                    predictor     = model,
                    overlap       = 0.5,
                    mode          = "gaussian",
                )

            val_outputs_post = [post_pred(i) for i in val_outputs]
            val_labels_post  = [post_label(i) for i in val_labels]
            dice_metric(y_pred=val_outputs_post, y=val_labels_post)

            pred_class  = torch.argmax(val_outputs, dim=1).cpu().numpy().flatten()
            label_class = val_labels.cpu().numpy().flatten()
            fg_mask     = label_class > 0
            all_preds.append(pred_class[fg_mask])
            all_labels_list.append(label_class[fg_mask])

    mean_dice      = dice_metric.aggregate().item()
    all_preds      = np.concatenate(all_preds)
    all_labels_arr = np.concatenate(all_labels_list)

    print(f"\nVal Dice: {mean_dice:.4f} | Best: {best_dice:.4f}")
    print(f"Voxel Accuracy: {accuracy_score(all_labels_arr, all_preds):.4f}")
    print(classification_report(all_labels_arr, all_preds, labels=[1,2,3], target_names=["TC","WT","ET"], digits=4))
    print(f"Confusion Matrix:\n{confusion_matrix(all_labels_arr, all_preds)}\n")

    if mean_dice > best_dice:
        best_dice  = mean_dice
        no_improve = 0
        torch.save({
            "epoch":       epoch + 1,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "best_dice":   best_dice,
        }, CHECKPOINT_PATH)
        print(f"Checkpoint saved (Dice: {best_dice:.4f}) at epoch {epoch+1}\n")
    else:
        no_improve += 1
        print(f"No improvement ({no_improve}/{PATIENCE})\n")
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

print(f"\nDone. Total: {time_elapsed()/60:.1f} min | Best Dice: {best_dice:.4f}")

# ---- verify checkpoint after training ----
if os.path.exists(CHECKPOINT_PATH):
    ck   = torch.load(CHECKPOINT_PATH, map_location="cpu")
    size = os.path.getsize(CHECKPOINT_PATH) / 1e6
    print(f"Checkpoint: {size:.1f} MB | Epoch: {ck['epoch']} | Dice: {ck['best_dice']:.4f}")
    print("Go to the Output tab in this notebook and download best_model.pth before closing the session.")

In [ ]:
import os
import shutil
import torch
 
CHECKPOINT_PATH = "/kaggle/working/best_model.pth"
 
# Verify the file exists and is not corrupted
assert os.path.exists(CHECKPOINT_PATH), "Checkpoint not found at /kaggle/working/best_model.pth"
 
ck   = torch.load(CHECKPOINT_PATH, map_location="cpu")
size = os.path.getsize(CHECKPOINT_PATH) / 1e6
 
print(f"Checkpoint verified")
print(f"Size  : {size:.1f} MB")
print(f"Epoch : {ck['epoch']}")
print(f"Dice  : {ck['best_dice']:.4f}")
print(f"\nCopy 1: /kaggle/working/best_model.pth — OK")
 
# Save 2 — duplicate with epoch+dice in filename so you know which run it is
versioned_name = f"/kaggle/working/swinunetr48_epoch{ck['epoch']}_dice{ck['best_dice']:.4f}.pth"
shutil.copy(CHECKPOINT_PATH, versioned_name)
print(f"Copy 2: {versioned_name} — OK")
 
# Save 3 — write a small metadata text file so you never forget what this checkpoint is
meta_path = "/kaggle/working/checkpoint_info.txt"
with open(meta_path, "w") as f:
    f.write(f"Architecture : SwinUNETR-48\n")
    f.write(f"Dataset      : BraTS 2023 GLI\n")
    f.write(f"Crop size    : (128, 128, 128)\n")
    f.write(f"Best epoch   : {ck['epoch']}\n")
    f.write(f"Best Dice    : {ck['best_dice']:.4f}\n")
    f.write(f"Loss         : DiceFocalLoss (gamma=2.0)\n")
    f.write(f"Optimizer    : AdamW lr=1e-4\n")
    f.write(f"Scheduler    : OneCycleLR\n")
print(f"Copy 3: {meta_path} — OK")
 
print("\nNow go to: Notebook top-right -> Save Version -> Save & Run All (Commit)")
print("This commits all files in /kaggle/working/ to the Output tab permanently.")
print("After commit finishes, your model is safe even if the session closes.")
 

In [ ]:
import os

# Write a dataset metadata file
os.makedirs("/kaggle/working/swinunetr_brats", exist_ok=True)

import shutil
shutil.copy(
    "/kaggle/working/best_model.pth",
    "/kaggle/working/swinunetr_brats/best_model.pth"
)

# Write dataset-metadata.json so Kaggle recognizes it as a dataset
import json
meta = {
    "title": "swinunetr-brats2023",
    "id": "dravyareddy/swinunetr-brats2023",
    "licenses": [{"name": "CC0-1.0"}]
}
with open("/kaggle/working/swinunetr_brats/dataset-metadata.json", "w") as f:
    json.dump(meta, f)

print("Folder ready for upload")

In [3]:
import torch
from monai.networks.nets import SwinUNETR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SwinUNETR(
    img_size          = (128, 128, 128),
    in_channels       = 4,
    out_channels      = 4,
    feature_size      = 48,
    use_checkpoint    = False,
    spatial_dims      = 3,
    dropout_path_rate = 0.1,
).to(device)

checkpoint = torch.load(
    "/kaggle/input/datasets/dravyareddy/swinunetr-brats2023/best_model.pth",
    map_location=device
)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"Model loaded | Epoch {checkpoint['epoch']} | Dice {checkpoint['best_dice']:.4f}")

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.nets.swin_unetr SwinUNETR.__init__:img_size: Argument `img_size` has been deprecated since version 1.3. It will be removed in version 1.5. The img_size argument is not required anymore and checks on the input size are run during forward().
  warn_deprecated(argname, msg, warning_category)


Model loaded | Epoch 70 | Dice 0.8586


In [4]:
import os, torch, numpy as np
from glob import glob
from scipy import ndimage
from monai.data import Dataset
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    Orientationd, NormalizeIntensityd, CropForegroundd,
    ConcatItemsd, DeleteItemsd, Lambdad, ToTensord, SpatialPadd
)

BASE = "/kaggle/input/datasets/dravyareddy/brats-2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
MODALITY_KEYS = ["flair", "t1", "t1ce", "t2"]

def brats_relabel(label):
    label[label == 4] = 3
    return label

inference_transforms = Compose([
    LoadImaged(keys=MODALITY_KEYS + ["label"]),
    EnsureChannelFirstd(keys=MODALITY_KEYS + ["label"]),
    Lambdad(keys=["label"], func=brats_relabel),
    ConcatItemsd(keys=MODALITY_KEYS, name="image", dim=0),
    DeleteItemsd(keys=MODALITY_KEYS),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 128)),
    ToTensord(keys=["image", "label"])
])

def get_file(case, keyword):
    for f in os.listdir(case):
        if keyword in f and (f.endswith(".nii") or f.endswith(".nii.gz")):
            return os.path.join(case, f)
    return None

def is_valid_case(case_dir):
    for kw in ["t2f", "t1n", "t1c", "t2w", "seg"]:
        path = get_file(case_dir, kw)
        if path is None or os.path.getsize(path) == 0:
            return False
    return True

def run_single_case(case_id):
    global mri_numpy
    case_dir  = os.path.join(BASE, case_id)
    if not os.path.exists(case_dir):
        print(f"Case {case_id} not found"); return None
    if not is_valid_case(case_dir):
        print(f"Case {case_id} has empty files"); return None

    case_dict = {
        "flair": get_file(case_dir, "t2f"),
        "t1":    get_file(case_dir, "t1n"),
        "t1ce":  get_file(case_dir, "t1c"),
        "t2":    get_file(case_dir, "t2w"),
        "label": get_file(case_dir, "seg"),
    }

    print(f"Loading {case_id}...")
    ds   = Dataset([case_dict], inference_transforms)
    item = ds[0]
    inp  = item["image"].unsqueeze(0).to(device).float()
    gt   = item["label"].squeeze().cpu().numpy()

    print("Running segmentation...")
    model.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            output = sliding_window_inference(
                inp, roi_size=(128,128,128),
                sw_batch_size=2, predictor=model,
                overlap=0.5, mode="gaussian",
            )

    seg_mask  = torch.argmax(output, dim=1).squeeze().cpu().numpy()
    mri_numpy = inp.squeeze(0).cpu().numpy()

    del inp, output
    torch.cuda.empty_cache()

    print("Extracting features...")
    features, query = extract_all_features(seg_mask, mri_numpy)
    features["case_id"] = case_id

    print(f"\nCase       : {case_id}")
    print(f"Location   : {features.get('hemisphere','N/A')} hemisphere, "
          f"{features.get('coronal_location','N/A')} {features.get('axial_location','N/A')}")
    print(f"WT volume  : {features.get('wt_volume_cc','N/A')} cc")
    print(f"TC volume  : {features.get('tc_volume_cc','N/A')} cc")
    print(f"ET volume  : {features.get('et_volume_cc','N/A')} cc")
    print(f"NCR volume : {features.get('ncr_volume_cc','N/A')} cc")
    print(f"ED volume  : {features.get('ed_volume_cc','N/A')} cc")
    print(f"ET/TC ratio: {features.get('et_tc_ratio','N/A')}")
    print(f"Sphericity : {features.get('sphericity','N/A')}")
    print(f"Enhancement: {features.get('enhancement_ratio','N/A')}")
    print(f"\nQuery:\n{query}")

    return {
        "case_id":  case_id,
        "seg_mask": seg_mask,
        "gt_mask":  gt,
        "features": features,
        "query":    query,
    }

all_cases   = sorted(glob(os.path.join(BASE, "*")))
valid_cases = [os.path.basename(c) for c in all_cases if is_valid_case(c)]
print(f"Valid cases: {len(valid_cases)}")

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.croppad.dictionary CropForegroundd.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


Valid cases: 989


In [5]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import nibabel as nib
from skimage.transform import resize
 
def plot_segmentation_overlay(mri_numpy, seg_mask, case_id, raw_flair_path=None):
    """
    mri_numpy      : (4, H, W, D) numpy array from inference
    seg_mask       : (H, W, D)    numpy array of predicted labels 0/1/2/3
    case_id        : string case identifier for title and filename
    raw_flair_path : path to original .nii file for sharper background
                     if None falls back to mri_numpy FLAIR channel
    """
    if raw_flair_path is not None and os.path.exists(raw_flair_path):
        raw = nib.load(raw_flair_path).get_fdata()
        # match shape to seg_mask — use mri_numpy if shapes differ
        if raw.shape == seg_mask.shape:
            flair = raw
        else:
            flair = mri_numpy[0]
    else:
        flair = mri_numpy[0]

    # normalize for better contrast
    p1, p99 = np.percentile(flair[flair > 0], [1, 99])
    flair    = np.clip(flair, p1, p99)
    flair    = (flair - p1) / (p99 - p1 + 1e-8)
 
    tumour = (seg_mask > 0).astype(int)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Case: {case_id}", fontsize=13, fontweight="bold")
    axis_names = ["Axial", "Coronal", "Sagittal"]
 
    for ax_idx, (ax, axis) in enumerate(zip(axes, [0, 1, 2])):
        if axis == 0:
            counts = tumour.sum(axis=(1, 2))
            sl     = int(np.argmax(counts))
            fl     = flair[sl, :, :]
            mk     = seg_mask[sl, :, :]
        elif axis == 1:
            counts = tumour.sum(axis=(0, 2))
            sl     = int(np.argmax(counts))
            fl     = flair[:, sl, :]
            mk     = seg_mask[:, sl, :]
        else:
            counts = tumour.sum(axis=(0, 1))
            sl     = int(np.argmax(counts))
            fl     = flair[:, :, sl]
            mk     = seg_mask[:, :, sl]
 
        if mk.shape != fl.shape:
            mk = resize(
                mk, fl.shape,
                order=0,
                preserve_range=True,
                anti_aliasing=False
            ).astype(np.int32)
 
        ov = np.zeros((*fl.shape, 4))
        ov[mk == 1] = [1.0, 0.2, 0.2, 0.65]   # NCR — red
        ov[mk == 2] = [1.0, 1.0, 0.2, 0.40]   # ED  — yellow
        ov[mk == 3] = [0.2, 0.4, 1.0, 0.70]   # ET  — blue
 
        ax.imshow(fl.T, cmap="gray", origin="lower")
        ax.imshow(ov.transpose(1, 0, 2), origin="lower")
        ax.set_title(axis_names[ax_idx], fontsize=11)
        ax.axis("off")
 
    patches = [
        mpatches.Patch(color=[1.0, 0.2, 0.2], label="NCR (Necrotic Core)"),
        mpatches.Patch(color=[1.0, 1.0, 0.2], label="ED (Edema)"),
        mpatches.Patch(color=[0.2, 0.4, 1.0], label="ET (Enhancing Tumor)"),
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3,
               fontsize=10, framealpha=0.9)
    plt.tight_layout(rect=[0, 0.06, 1, 1])
 
    save_path = f"/kaggle/working/{case_id}_overlay.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
 

In [6]:
from scipy import ndimage
 
def extract_all_features(seg_mask, mri_tensor=None, affine=None, voxel_volume_cc=0.001):
    """
    seg_mask   : (H, W, D) numpy array, labels 0/1/2/3
    mri_tensor : (4, H, W, D) numpy array — FLAIR, T1, T1CE, T2
    returns    : (features dict, clinical query string)
    """
    features = {}
 
    # BraTS label definitions
    # 1 = NCR (necrotic core)
    # 2 = ED  (edema)
    # 3 = ET  (enhancing tumor)
    # TC = NCR + ET
    # WT = NCR + ED + ET
 
    ncr_vox = int(np.sum(seg_mask == 1))
    ed_vox  = int(np.sum(seg_mask == 2))
    et_vox  = int(np.sum(seg_mask == 3))
    tc_vox  = ncr_vox + et_vox
    wt_vox  = ncr_vox + ed_vox + et_vox
 
    tc_vol  = tc_vox  * voxel_volume_cc
    wt_vol  = wt_vox  * voxel_volume_cc
    et_vol  = et_vox  * voxel_volume_cc
    ed_vol  = ed_vox  * voxel_volume_cc
    ncr_vol = ncr_vox * voxel_volume_cc
 
    features["wt_volume_cc"]  = round(wt_vol,  2)
    features["tc_volume_cc"]  = round(tc_vol,  2)
    features["et_volume_cc"]  = round(et_vol,  2)
    features["ncr_volume_cc"] = round(ncr_vol, 2)
    features["ed_volume_cc"]  = round(ed_vol,  2)
    features["et_tc_ratio"]   = round(et_vol  / tc_vol,  3) if tc_vol  > 0 else 0
    features["ncr_tc_ratio"]  = round(ncr_vol / tc_vol,  3) if tc_vol  > 0 else 0
    features["ed_wt_ratio"]   = round(ed_vol  / wt_vol,  3) if wt_vol  > 0 else 0
 
    # spatial location — RAS space: low W = right, high W = left
    # spatial location using RAS world coordinates
    if wt_vox > 0:

        coords = np.argwhere(seg_mask > 0)
        centroid = coords.mean(axis=0)

        H, W, D = seg_mask.shape

        if affine is not None:

        # convert voxel centroid -> world RAS
            centroid_h = np.append(centroid, 1)
            world_centroid = affine @ centroid_h

            x, y, z = world_centroid[:3]

        # RAS:
        # x < 0 = right hemisphere
        # x > 0 = left hemisphere
            features["hemisphere"] = ("right" if x < 0 else "left")

        # y axis: anterior/posterior
            features["coronal_location"] = (
                "frontal" if y > 120 else
                "parieto-temporal"
            )

        # z axis: inferior/superior
            features["axial_location"] = (
                "inferior" if z < 80 else
                "middle" if z < 160 else
                "superior"
            )

        else:

        # fallback if affine unavailable
            features["hemisphere"] = (
                "right" if centroid[1] < W/2 else "left"
            )

            features["axial_location"] = (
                "inferior" if centroid[0] < H/3 else
                "middle" if centroid[0] < 2*H/3 else
                "superior"
            )

            features["coronal_location"] = (
                "frontal" if centroid[2] < D/3 else
                "parieto-temporal" if centroid[2] < 2*D/3 else
                "occipital"
            )


    else:

        features["hemisphere"] = "unknown"
        features["axial_location"] = "unknown"
        features["coronal_location"] = "unknown"
    
 
    # shape features (Zwanenburg et al. 2020)
    if wt_vox > 0:
        wt_mask  = (seg_mask > 0).astype(np.uint8)
        bbox     = ndimage.find_objects(wt_mask)[0]
        bh = bbox[0].stop - bbox[0].start
        bw = bbox[1].stop - bbox[1].start
        bd = bbox[2].stop - bbox[2].start
        bbox_vol = bh * bw * bd * voxel_volume_cc
 
        features["solidity"]   = round(wt_vol / bbox_vol, 3) if bbox_vol > 0 else 0
        dims = sorted([bh, bw, bd])
        features["elongation"] = round(dims[2] / dims[0], 3) if dims[0] > 0 else 0
 
        grad     = np.gradient(wt_mask.astype(float))
        surf_vox = np.sum(np.sqrt(sum(g**2 for g in grad)) > 0.5)
        features["sphericity"] = round(
            (np.pi**(1/3) * (6 * wt_vox)**(2/3)) / surf_vox, 3
        ) if surf_vox > 0 else 0
 
    # intensity features
    if mri_tensor is not None:
        tc_mask = (seg_mask == 1) | (seg_mask == 3)
 
        for i, name in enumerate(["FLAIR", "T1", "T1CE", "T2"]):
            vol = mri_tensor[i]
            for region, cond in [
                ("wt", seg_mask > 0),
                ("tc", tc_mask),
                ("et", seg_mask == 3),
            ]:
                if np.sum(cond) < 10:
                    continue
                vox = vol[cond]
                features[f"{name}_{region}_mean"] = round(float(np.mean(vox)), 4)
                features[f"{name}_{region}_std"]  = round(float(np.std(vox)),  4)
 
        # enhancement ratio (Ellingson et al. 2017)
        normal_mask = seg_mask == 0
        et_mask     = seg_mask == 3
        if np.sum(normal_mask) > 100 and np.sum(et_mask) > 10:
            mean_normal = float(np.mean(np.clip(mri_tensor[2][normal_mask], 0, None)))
            mean_et     = float(np.mean(np.clip(mri_tensor[2][et_mask],     0, None)))
            features["enhancement_ratio"] = round(
                mean_et / mean_normal if mean_normal > 0.01 else 0, 3
            )
 
    query = (
        f"Glioma in {features['hemisphere']} hemisphere, "
        f"{features['coronal_location']} {features['axial_location']} region. "
        f"Whole tumor {features['wt_volume_cc']}cc, "
        f"tumor core {features['tc_volume_cc']}cc "
        f"(NCR {features['ncr_volume_cc']}cc + ET {features['et_volume_cc']}cc), "
        f"edema {features['ed_volume_cc']}cc. "
        f"ET/TC ratio {features['et_tc_ratio']}, "
        f"NCR/TC ratio {features['ncr_tc_ratio']}, "
        f"edema/WT ratio {features['ed_wt_ratio']}, "
        f"solidity {features.get('solidity', 'N/A')}, "
        f"sphericity {features.get('sphericity', 'N/A')}. "
        f"Enhancement ratio {features.get('enhancement_ratio', 'N/A')}. "
        f"What is the WHO grade, IDH status, MGMT methylation, "
        f"treatment protocol and prognosis?"
    )
 
    return features, query
 
 

In [ ]:
import os
from glob import glob

BASE = "/kaggle/input/datasets/dravyareddy/brats-2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

all_cases   = sorted(glob(os.path.join(BASE, "*")))
valid_cases = [os.path.basename(c) for c in all_cases if is_valid_case(c)]
invalid_cases = [os.path.basename(c) for c in all_cases if not is_valid_case(c)]

print(f"Total cases  : {len(all_cases)}")
print(f"Valid cases  : {len(valid_cases)}")
print(f"Invalid cases: {len(invalid_cases)}")

if invalid_cases:
    print(f"\nInvalid case IDs:")
    for c in invalid_cases:
        print(f"  {c}")

print(f"\nFirst 10 valid cases:")
for c in valid_cases[:10]:
    print(f"  {c}")

In [ ]:
case_id = "BraTS-GLI-00008-001"
 
# run inference + feature extraction
result = run_single_case(case_id)
 
# get raw FLAIR path for sharp overlay background
raw_flair = get_file(os.path.join(BASE, case_id), "t2f")
 
# plot segmentation overlay
plot_segmentation_overlay(
    mri_numpy      = mri_numpy,
    seg_mask       = result["seg_mask"],
    case_id        = case_id,
    raw_flair_path = raw_flair
)
  

In [7]:
 
import subprocess
subprocess.run(["pip", "install", "-q", "gradio"], check=True)
 
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
from skimage.transform import resize
import tempfile
import random
 
 
def build_overlay_image(mri_numpy, seg_mask, case_id, raw_flair_path=None):
    if raw_flair_path is not None and os.path.exists(raw_flair_path):
        raw = nib.load(raw_flair_path).get_fdata()
        flair = raw if raw.shape == seg_mask.shape else mri_numpy[0]
    else:
        flair = mri_numpy[0]
 
    p1, p99 = np.percentile(flair[flair > 0], [1, 99])
    flair    = np.clip(flair, p1, p99)
    flair    = (flair - p1) / (p99 - p1 + 1e-8)
 
    tumour = (seg_mask > 0).astype(int)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    fig.suptitle(f"Case: {case_id}", fontsize=12, fontweight="bold")
    axis_names = ["Axial", "Coronal", "Sagittal"]
 
    for ax, axis, name in zip(axes, [0, 1, 2], axis_names):
        if axis == 0:
            counts = tumour.sum(axis=(1, 2)); sl = int(np.argmax(counts))
            fl = flair[sl, :, :]; mk = seg_mask[sl, :, :]
        elif axis == 1:
            counts = tumour.sum(axis=(0, 2)); sl = int(np.argmax(counts))
            fl = flair[:, sl, :]; mk = seg_mask[:, sl, :]
        else:
            counts = tumour.sum(axis=(0, 1)); sl = int(np.argmax(counts))
            fl = flair[:, :, sl]; mk = seg_mask[:, :, sl]
 
        if mk.shape != fl.shape:
            mk = resize(mk, fl.shape, order=0, preserve_range=True,
                        anti_aliasing=False).astype(np.int32)
 
        ov = np.zeros((*fl.shape, 4))
        ov[mk == 1] = [1.0, 0.2, 0.2, 0.65]
        ov[mk == 2] = [1.0, 1.0, 0.2, 0.40]
        ov[mk == 3] = [0.2, 0.4, 1.0, 0.70]
 
        ax.imshow(fl.T, cmap="gray", origin="lower")
        ax.imshow(ov.transpose(1, 0, 2), origin="lower")
        ax.set_title(name, fontsize=10)
        ax.axis("off")
 
    plt.tight_layout()
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmp.name, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return tmp.name
 
 
def format_features_markdown(f):
    return f"""
| Metric | Value |
|---|---|
| **Location** | {f.get('hemisphere','N/A')} hemisphere, {f.get('coronal_location','N/A')} {f.get('axial_location','N/A')} |
| **Whole Tumor (WT)** | {f.get('wt_volume_cc','N/A')} cc |
| **Tumor Core (TC)** | {f.get('tc_volume_cc','N/A')} cc |
| **Enhancing Tumor (ET)** | {f.get('et_volume_cc','N/A')} cc |
| **Necrotic Core (NCR)** | {f.get('ncr_volume_cc','N/A')} cc |
| **Edema (ED)** | {f.get('ed_volume_cc','N/A')} cc |
| **ET/TC Ratio** | {f.get('et_tc_ratio','N/A')} |
| **Edema/WT Ratio** | {f.get('ed_wt_ratio','N/A')} |
| **Solidity** | {f.get('solidity','N/A')} |
| **Sphericity** | {f.get('sphericity','N/A')} |
| **Enhancement Ratio** | {f.get('enhancement_ratio','N/A')} |
"""
 
 
def run_ui_pipeline(case_id_input, pick_random):
    case_id = random.choice(valid_cases) if pick_random else case_id_input.strip()
 
    if case_id not in valid_cases:
        return None, f"Case `{case_id}` not found or invalid. Try one from the dropdown.", "", case_id
 
    result = run_single_case(case_id)
    if result is None:
        return None, "Inference failed for this case.", "", case_id
 
    raw_flair = get_file(os.path.join(BASE, case_id), "t2f")
    overlay_path = build_overlay_image(
        mri_numpy = mri_numpy,
        seg_mask  = result["seg_mask"],
        case_id   = case_id,
        raw_flair_path = raw_flair
    )
 
    features_md = format_features_markdown(result["features"])
    query_text  = result["query"]
 
    return overlay_path, features_md, query_text, case_id
 
 
with gr.Blocks(title="BraTS Brain Tumor Segmentation") as demo:
    gr.Markdown("# Brain Tumor Segmentation Viewer")
    gr.Markdown(
        "SwinUNETR model trained on BraTS 2023. Select a case ID from the dataset "
        "to run segmentation and view extracted radiomic features."
    )
 
    with gr.Row():
        case_dropdown = gr.Dropdown(
            choices = valid_cases,
            value   = valid_cases[0] if valid_cases else None,
            label   = "Select a BraTS Case ID",
            scale   = 3
        )
        random_btn = gr.Button("Random Case", scale=1)
        run_btn    = gr.Button("Run Segmentation", variant="primary", scale=1)
 
    case_label = gr.Textbox(label="Current Case", interactive=False)
 
    with gr.Row():
        overlay_output = gr.Image(label="Segmentation Overlay (Axial / Coronal / Sagittal)")
 
    with gr.Row():
        features_output = gr.Markdown(label="Extracted Features")
 
    with gr.Row():
        query_output = gr.Textbox(label="Generated Clinical Query (for RAG/LLM)", lines=4)
 
    def run_selected(case_id):
        return run_ui_pipeline(case_id, pick_random=False)
 
    def run_random():
        return run_ui_pipeline("", pick_random=True)
 
    run_btn.click(
        fn      = run_selected,
        inputs  = [case_dropdown],
        outputs = [overlay_output, features_output, query_output, case_label]
    )
 
    random_btn.click(
        fn      = run_random,
        inputs  = [],
        outputs = [overlay_output, features_output, query_output, case_label]
    )
 
demo.launch(share=True, debug=False)
 

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2f81373d2cd65fe183.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading BraTS-GLI-00048-001...
Running segmentation...


/tmp/ipykernel_129/1940223047.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:223: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:361: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interprete

Extracting features...

Case       : BraTS-GLI-00048-001
Location   : right hemisphere, frontal inferior
WT volume  : 22.65 cc
TC volume  : 0.0 cc
ET volume  : 0.0 cc
NCR volume : 0.0 cc
ED volume  : 22.65 cc
ET/TC ratio: 1.0
Sphericity : 0.718
Enhancement: N/A

Query:
Glioma in right hemisphere, frontal inferior region. Whole tumor 22.65cc, tumor core 0.0cc (NCR 0.0cc + ET 0.0cc), edema 22.65cc. ET/TC ratio 1.0, NCR/TC ratio 0.0, edema/WT ratio 1.0, solidity 0.043, sphericity 0.718. Enhancement ratio N/A. What is the WHO grade, IDH status, MGMT methylation, treatment protocol and prognosis?
Loading BraTS-GLI-00021-001...
Running segmentation...


/tmp/ipykernel_129/1940223047.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:223: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:361: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interprete

Extracting features...

Case       : BraTS-GLI-00021-001
Location   : left hemisphere, parieto-temporal middle
WT volume  : 124.43 cc
TC volume  : 40.72 cc
ET volume  : 20.62 cc
NCR volume : 20.11 cc
ED volume  : 83.7 cc
ET/TC ratio: 0.506
Sphericity : 0.556
Enhancement: 7.434

Query:
Glioma in left hemisphere, parieto-temporal middle region. Whole tumor 124.43cc, tumor core 40.72cc (NCR 20.11cc + ET 20.62cc), edema 83.7cc. ET/TC ratio 0.506, NCR/TC ratio 0.494, edema/WT ratio 0.673, solidity 0.201, sphericity 0.556. Enhancement ratio 7.434. What is the WHO grade, IDH status, MGMT methylation, treatment protocol and prognosis?
Loading BraTS-GLI-00051-000...
Running segmentation...


/tmp/ipykernel_129/1940223047.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:223: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:361: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interprete

Extracting features...

Case       : BraTS-GLI-00051-000
Location   : left hemisphere, parieto-temporal inferior
WT volume  : 111.64 cc
TC volume  : 22.35 cc
ET volume  : 18.83 cc
NCR volume : 3.52 cc
ED volume  : 89.29 cc
ET/TC ratio: 0.843
Sphericity : 0.673
Enhancement: 13.778

Query:
Glioma in left hemisphere, parieto-temporal inferior region. Whole tumor 111.64cc, tumor core 22.35cc (NCR 3.52cc + ET 18.83cc), edema 89.29cc. ET/TC ratio 0.843, NCR/TC ratio 0.157, edema/WT ratio 0.8, solidity 0.312, sphericity 0.673. Enhancement ratio 13.778. What is the WHO grade, IDH status, MGMT methylation, treatment protocol and prognosis?
